# 00 — Môi trường và cấu hình

Ba client **UNSW + BoT + CIC**; 39 feature. CIC giữ đúng 1.000.000 mẫu thật (500.000 benign + 500.000 attack). Mọi mô hình dùng **CSV raw → Log → Z-score riêng từng client → clip [-8,8]**. Mean/std chỉ fit trên tập fit; đóng băng khi dùng validation/test. Split và nhãn được giữ nguyên. Kết quả mới ở `runs/main/`.

In [ ]:
from pathlib import Path
import os, sys, json
if os.name == 'nt':
    raise RuntimeError('Mở start_notebooks.cmd để dùng kernel WSL /opt/flids-tf.')
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p/'notebook_workflow.py').is_file() and (p/'configs/tuning.json').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('Mở notebook trong FL_IDS.')
os.chdir(PROJECT)
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
CONFIG = PROJECT/'configs/tuning.json'
from notebook_workflow import settings, budget
C = settings(CONFIG)
RUN_ROOT = PROJECT/C['run_directory']
print('Project:', PROJECT)
print('Python:', sys.executable)
print('Run:', RUN_ROOT)


In [ ]:
from common import config
for name, relative in config()['datasets'].items():
    path = PROJECT/'dataset'/relative
    print(name, path, path.is_file())
    assert path.is_file()
display(budget(CONFIG))

In [ ]:
from runtime import configure
display(configure())

Chạy lần lượt **01 → 02 → 03 → 04**: chuẩn bị dữ liệu, tuning, huấn luyện, tổng hợp kết quả. Ngân sách chung: FL 15 round × 3 local epoch; Local/Centralized tối đa 45 epoch, early stopping val_loss patience 5.